# RSICD Patch-Level Raster Labeling (Part B)
### Generating semantic 7×7 patch labels with Qwen3.8-27B via Groq

This notebook consumes the **structured dataset** produced by Part A and generates semantic raster labels
for every cell of a fixed 7×7 image grid. Qwen3.8-27B is used as a vision-language model (VLM) labeler,
with the scene-specific candidate vocabulary from Part A constraining every predicted label.

**Method:** each RSICD image is upscaled and overlaid with a visible 7×7 grid numbered 0–48. The model
labels each marked cell using the image, scene class, reference captions, and restricted candidate
vocabulary. Structured JSON output ensures every returned label belongs to the allowed vocabulary.

**Scope:** this run processes 8 visually distinct RSICD scene classes rather than all 31, keeping the
labeling and manual-review workload tractable within the project timeline.

**Pipeline:**
1. Setup — imports
2. Load Groq API keys with rotation
3. Configure Qwen3.8-27B and test clients
4. Load the Part A patch-labeling dataset
5. Build the 7×7 grid overlay
6. Construct the labeling prompt
7. Define the structured JSON output schema
8. Run the core VLM labeling call
9. Validate model output
10. Add key rotation, backoff, and fallback handling
11. Visualize raster-labeling results
12. Select the target scene classes
13. Process each target class in resumable batches
14. Run one class at a time
15. Migrate legacy output format if required

**Inputs:** `patch_labeling_dataset.json` from Part A + RSICD images.

**Outputs:** one JSON file per target scene class in `/kaggle/working/qwen_labels/`, mapping each
filename to its 49 cell labels and `needs_review` status.


In [ ]:
!pip install -q groq

## 1. Setup


In [ ]:
import os
import json
import time
from io import BytesIO
import base64

from PIL import Image, ImageDraw
from groq import Groq
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## 2. Load Groq API Key Pool

Groq enforces per-key rate limits. Multiple Kaggle Secrets (`GROQ_API_KEY_1` ... `GROQ_API_KEY_28`)
are loaded so requests can rotate across keys, increasing effective throughput and providing failover
when an individual key is rate-limited or temporarily unavailable.


In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GROQ_API_KEYS = []

for i in range(1, 29):
    try:
        key = secrets.get_secret(f"GROQ_API_KEY_{i}")
        GROQ_API_KEYS.append(key)
    except Exception:
        pass

print(f"Loaded {len(GROQ_API_KEYS)} API key(s)")
assert len(GROQ_API_KEYS) > 0, "No GROQ_API_KEY_n secrets found -- check Kaggle Secrets setup"

## 3. Model Configuration & Client Test

Configure Qwen3.8-27B and verify that every loaded Groq client can reach the model before real labeling
starts. Failures report only the key index, never the raw API key.


In [ ]:
MODEL_NAME = "qwen/qwen3.8-27b"  # change to "qwen/qwen3.6-27b" if preferred

clients = [Groq(api_key=k) for k in GROQ_API_KEYS]
rotation_state = {"idx": 0}

for i in range(0, len(clients)):
    try:
        test_response = clients[i].chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": "Reply with just the word: ok"}],
            reasoning_effort="none",
        )
        print("client num", i, test_response.choices[0].message.content)
    except Exception as e:
        print("client num", i, "FAILED:", e)

## 4. Load Part A Patch-Labeling Dataset

Load `patch_labeling_dataset.json` produced by Part A. Each record contains the image filename, RSICD
scene class, 5 reference captions, scene-specific candidate vocabulary and indices, and the 49 fixed
grid-cell definitions.


In [ ]:
PART_A_DIR = "/kaggle/input/datasets/vedantbarhate007/eric-raster-labelling-dataset/Raster_Labeling_Dataset"
IMG_DIR = "/kaggle/input/datasets/vedantbarhate007/rsicd-eric/RSICD_optimal-master/RSICD_images"

PATCH_DATASET_FILE = os.path.join(PART_A_DIR, "patch_labeling_dataset.json")

with open(PATCH_DATASET_FILE, "r") as f:
    patch_dataset = json.load(f)

print("Loaded records:", len(patch_dataset))
print(json.dumps(patch_dataset[0], indent=2)[:800])

## 5. Build the 7×7 Grid Overlay

Create a visible 7×7 white grid with cell IDs 0–48 on an upscaled copy of each image. The overlay gives
the VLM an explicit spatial partition so it can refer to marked regions directly rather than relying
only on implicit spatial reasoning.


In [ ]:
def draw_grid_overlay(image, grid_size=7, upscale_to=672):
    img = image.resize((upscale_to, upscale_to)).convert("RGB")
    draw = ImageDraw.Draw(img)
    cell_size = upscale_to // grid_size

    for i in range(1, grid_size):
        draw.line([(i * cell_size, 0), (i * cell_size, upscale_to)], fill="white", width=2)
        draw.line([(0, i * cell_size), (upscale_to, i * cell_size)], fill="white", width=2)

    for row in range(grid_size):
        for col in range(grid_size):
            cell_id = row * grid_size + col
            x = col * cell_size + 4
            y = row * cell_size + 4
            draw.text((x, y), str(cell_id), fill="yellow")

    return img

## 6. Construct the Labeling Prompt

Build the per-image prompt from the scene class, 5 reference captions, and the restricted candidate
vocabulary. The prompt enforces one label per cell, uses a majority-area rule, and asks the model to
consider neighboring cells when an object spans multiple regions.


In [ ]:
def build_prompt(record):
    captions = chr(10).join('- ' + c for c in record['captions'])
    terms = ", ".join(record['candidate_terms'])

    return f"""Remote sensing image, scene type: "{record['scene']}". Overlaid with a 7x7 grid, cells numbered 0-48, left-to-right then top-to-bottom.

Reference captions:
{captions}

Rules:
1. Label each of the 49 cells with exactly ONE term from this list: {terms}
2. Pick the label that best matches what covers MOST of that cell's area.
3. If a cell's content is unclear or ambiguous on its own, check its left, right, top, and bottom neighboring cells for context before deciding -- real objects (buildings, roads, water) usually span multiple adjacent cells.
4. Do not guess a label that has no visual evidence in the cell or its neighbors.

Return a label for all 49 cells, no cell skipped."""

## 7. Define the Structured Output Schema

Create an enum-constrained JSON schema from the image's own `candidate_terms`. The schema restricts
the model to valid vocabulary terms and ensures the response has the expected cell-level structure.


In [ ]:
def build_schema(candidate_terms):
    return {
        "type": "json_schema",
        "json_schema": {
            "name": "patch_labels",
            "schema": {
                "type": "object",
                "properties": {
                    "cells": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "cell_id": {"type": "integer"},
                                "label": {"type": "string", "enum": candidate_terms}
                            },
                            "required": ["cell_id", "label"]
                        }
                    }
                },
                "required": ["cells"]
            }
        }
    }

## 8. Run the Core VLM Labeling Call

Encode the grid-overlaid image as base64 and send it to Qwen3.8-27B through a Groq client together
with the labeling prompt and structured output schema. `reasoning_effort="high"` is used for harder
spatial decisions, while `temperature=0` keeps the labeling deterministic.


In [ ]:
def label_image(client, record, img_dir, model_name=MODEL_NAME):
    image = Image.open(os.path.join(img_dir, record["filename"]))
    overlay = draw_grid_overlay(image)

    buf = BytesIO()
    overlay.save(buf, format="PNG")
    image_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    image_data_uri = f"data:image/png;base64,{image_b64}"

    prompt = build_prompt(record)

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": image_data_uri}},
                ],
            }
        ],
        response_format=build_schema(record["candidate_terms"]),
        reasoning_effort="high",
        temperature=0
    )

    return json.loads(response.choices[0].message.content)

## 9. Validate Model Output

A schema-valid response can still be structurally incomplete. Validate that all 49 cells are present,
cell IDs are unique and within 0–48, and every label belongs to the image's actual candidate vocabulary.


In [ ]:
def validate_result(result, record):
    if "cells" not in result:
        return False

    cells = result["cells"]
    if len(cells) != 49:
        return False

    seen_ids = set()
    valid_labels = set(record["candidate_terms"])

    for c in cells:
        if "cell_id" not in c or "label" not in c:
            return False
        if c["cell_id"] in seen_ids:
            return False
        seen_ids.add(c["cell_id"])
        if c["label"] not in valid_labels:
            return False

    return seen_ids == set(range(49))

## 10. Robust Labeling — Rotation, Backoff & Fallback

On an API error or invalid result, rotate to the next Groq key and retry. After a full key cycle,
apply exponential backoff (`base_backoff × 2^(cycle-1)`, capped at `max_backoff`). If all configured
cycles fail, return a placeholder result and mark the image for review instead of blocking the batch.


In [ ]:
def label_image_robust(record, img_dir, clients, rotation_state, model_name=MODEL_NAME,
                        base_backoff=5, max_backoff=120, max_cycles=1):
    attempt = 0

    while True:
        client_idx = rotation_state["idx"] % len(clients)
        client = clients[client_idx]

        try:
            result = label_image(client, record, img_dir, model_name)

            if validate_result(result, record):
                rotation_state["idx"] += 1
                return result
            else:
                print(f"  [{record['filename']}] invalid/incomplete result on key {client_idx}, retrying...")

        except Exception as e:
            print(f"  [{record['filename']}] error on key {client_idx}: {e}")

        rotation_state["idx"] += 1
        attempt += 1

        if attempt % len(clients) == 0:
            cycle = attempt // len(clients)

            if cycle >= max_cycles:
                print(f"  [{record['filename']}] FAILED after {cycle} cycles -- "
                      f"filling with placeholder label for manual review")
                placeholder = record["candidate_terms"][0]
                fallback_cells = [{"cell_id": i, "label": placeholder} for i in range(49)]
                return {"cells": fallback_cells, "_needs_review": True}

            backoff = min(base_backoff * (2 ** (cycle - 1)), max_backoff)
            print(f"  Cycled through all {len(clients)} keys without success, backing off {backoff}s...")
            time.sleep(backoff)
        else:
            time.sleep(5)

## 11. Visualize Raster-Labeling Results

Display the original image with its grid beside the generated abstract raster. The visualization is
used for spot-checking individual predictions before trusting a larger batch.


In [ ]:
def visualize_result(record, result, img_dir, grid_size=7, upscale_to=672):
    labels_by_id = {c["cell_id"]: c["label"] for c in result["cells"]}
    image = Image.open(os.path.join(img_dir, record["filename"])).resize((upscale_to, upscale_to)).convert("RGB")
    cell_size = upscale_to // grid_size

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    axes[0].imshow(image)
    axes[0].set_title(f"Original + Grid Overlay\n{record['filename']}")
    for i in range(1, grid_size):
        axes[0].axhline(i * cell_size, color="white", linewidth=1)
        axes[0].axvline(i * cell_size, color="white", linewidth=1)
    axes[0].axis("off")

    unique_labels = sorted(set(labels_by_id.values()))
    cmap = plt.get_cmap("tab20", len(unique_labels))
    color_map = {label: cmap(i) for i, label in enumerate(unique_labels)}

    axes[1].set_xlim(0, grid_size)
    axes[1].set_ylim(0, grid_size)
    axes[1].invert_yaxis()
    axes[1].set_title(f"{record['scene']}, raster labels\n{record['filename']}")
    axes[1].axis("off")

    for cell_id, label in labels_by_id.items():
        row, col = divmod(cell_id, grid_size)
        color = color_map[label]
        axes[1].add_patch(patches.Rectangle(
            (col, row), 1, 1, facecolor=color, edgecolor="white", linewidth=2
        ))
        axes[1].text(col + 0.5, row + 0.5, label, ha="center", va="center", fontsize=6)

    plt.tight_layout()
    plt.show()

### Quick Test — Single Image

Run this before a batch, especially after changing the model or prompt, to confirm that the complete
labeling and visualization pipeline behaves as expected.


In [ ]:
test_record = patch_dataset[200]
test_image = Image.open(os.path.join(IMG_DIR, test_record["filename"]))
overlay = draw_grid_overlay(test_image)
display(overlay)

In [ ]:
result = label_image_robust(test_record, IMG_DIR, clients, rotation_state)
visualize_result(test_record, result, IMG_DIR)

Inspect the raw structured output for this test image:


In [ ]:
result

## 12. Select the Target Scene Classes

Labeling all 31 RSICD scene classes would require roughly 535k patch labels and is not tractable within
the project timeline. This run is therefore scoped to 8 visually distinct classes, avoiding near-duplicate
scene types so the eventual raster-fusion ablation measures a meaningful effect.

Target classes:

`airport, baseballfield, beach, mediumresidential, pond, river, sparseresidential, stadium`

The same 8-class scope should be used for downstream captioning ablations so comparisons remain consistent.


In [ ]:
TARGET_CLASSES = {
    "airport", "baseballfield", "beach", "mediumresidential",
    "pond", "river", "sparseresidential", "stadium"
}

target_records = [r for r in patch_dataset if r["scene"] in TARGET_CLASSES]
print(f"Total records across target classes: {len(target_records)}")

by_scene = {}
for r in target_records:
    by_scene.setdefault(r["scene"], []).append(r)

for scene, records in sorted(by_scene.items()):
    print(f"{scene}: {len(records)} images")

## 13. Process Target Classes in Resumable Batches

Create one output file per scene class at `/kaggle/working/qwen_labels/<scene>.json`. Each file maps
`filename` to `{cells, needs_review}`. The batch processor checkpoints every 10 images and skips
already-completed filenames, allowing a class to be safely interrupted and resumed.


In [ ]:
OUTPUT_DIR = "/kaggle/working/qwen_labels"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def process_scene_class(scene_name, records, img_dir, clients, rotation_state):
    out_path = os.path.join(OUTPUT_DIR, f"{scene_name}.json")

    if os.path.exists(out_path):
        with open(out_path) as f:
            scene_results = json.load(f)
    else:
        scene_results = {}

    done_filenames = set(scene_results.keys())
    remaining = [r for r in records if r["filename"] not in done_filenames]

    print(f"\n=== {scene_name}: {len(done_filenames)} done, {len(remaining)} remaining ===")

    for i, record in enumerate(remaining):
        start_time = time.strftime("%H:%M:%S")
        print(f"[{start_time}] labeling {record['filename']} ({i + 1}/{len(remaining)})...", end=" ", flush=True)

        result = label_image_robust(record, img_dir, clients, rotation_state)
        needs_review = result.pop("_needs_review", False)
        scene_results[record["filename"]] = {
            "cells": result["cells"],
            "needs_review": needs_review
        }

        done_time = time.strftime("%H:%M:%S")
        status = "FAILED-PLACEHOLDER" if needs_review else "done"
        print(f"{status} [{done_time}]")

        time.sleep(5)

        if (i + 1) % 10 == 0:
            with open(out_path, "w") as f:
                json.dump(scene_results, f)
            print(f"  checkpoint saved: {i + 1}/{len(remaining)}")

    with open(out_path, "w") as f:
        json.dump(scene_results, f)

    print(f"  {scene_name} complete: {len(scene_results)} total images labeled")

### Run One Class at a Time

Change `scene_name` and run the batch processor once for each of the 8 target classes. The output file
is preserved between runs, so interrupted classes can be resumed without repeating completed images.


In [ ]:
scene_name = "pond"  # change this each run: airport, baseballfield, beach, mediumresidential, pond, river, sparseresidential, stadium
process_scene_class(scene_name, by_scene[scene_name], IMG_DIR, clients, rotation_state)

## 14. Utility — Legacy Output Format Migration

This optional migration is only required for output files from an earlier run that stored each result
as a flat list instead of the current `{"cells": ..., "needs_review": ...}` wrapper. Leave the cell
commented unless an old output file causes a format mismatch such as `KeyError`.


In [ ]:
# import json
#
# for scene_file in os.listdir(OUTPUT_DIR):
#     path = os.path.join(OUTPUT_DIR, scene_file)
#     with open(path) as f:
#         data = json.load(f)
#
#     changed = False
#     for fname, entry in data.items():
#         if isinstance(entry, list):  # old flat-list format -> wrap it
#             data[fname] = {"cells": entry, "needs_review": False}
#             changed = True
#
#     if changed:
#         with open(path, "w") as f:
#             json.dump(data, f)
#         print(f"migrated {scene_file}")